# PhiState-MoE — Resume the real 10K model on Colab's L4

**This notebook survives runtime resets because its instructions live in GitHub.** `/content` itself is temporary and will be rebuilt on every new session.

1. Select **Runtime → Change runtime type → L4 GPU**.
2. Choose **Runtime → Run all**. Enter your two **private archive passwords** when asked (do not share them). The final checkpoint and newly re-keyed dataset archives may have **different passwords**.
3. The notebook restores your public source code and encrypted 10K model and re-keyed dataset from the existing GitHub Release, verifies the exact corpus fingerprint and optimizer, performs a **20-step real-data GPU pilot**, then continues to **10,250**.
4. Approve a normal Colab **Google Drive mount** at the end to save checkpoints before disconnecting.

Your 10,000-step checkpoint is not modified. The release contains public encrypted archives; never add passwords, private checkpoints, or datasets to the public repository. If a run fails, its real error appears in the cell output and its local log. These tests use the original dataset and are not sealed-test evaluation.


In [ ]:
# 1 — Restore everything after a Colab runtime reset. RUN THIS IN COLAB.
import sys, json, shutil, subprocess, gc
from pathlib import Path
from getpass import getpass
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > L4 GPU, then reconnect.")
print("CUDA:", torch.cuda.get_device_name(0), flush=True)

ROOT = Path("/content/phistate_l4")
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / "quantoniumoslinux"
LAB = REPO / "runtime/experiments/phi_runtime_lab"
if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
elif not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Anharmoniclabs/quantoniumoslinux.git",
                    str(REPO)], check=True)
else:
    raise RuntimeError(f"Partial repository at {REPO}; investigate before proceeding.")
for src in ("moe_model.py", "moe_train.py", "pretrain_continue_10k.py"):
    assert (LAB / src).is_file(), f"Missing GitHub source {src}"

if shutil.which("7z") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "7zip"], check=True)

RELEASE = ("https://github.com/Anharmoniclabs/quantoniumoslinux/"
           "releases/download/phistate-colab-probe-v1/")
ARCHIVE_SIZE = {"phi_final10k.7z": 141204483}
ARCHIVES = ROOT / "archives"
ARCHIVES.mkdir(exist_ok=True)

def restore_encrypted(asset, dest, expected):
    if expected.is_file():
        print("Already extracted:", expected, flush=True)
        return
    dest.mkdir(parents=True, exist_ok=True)
    archive = ARCHIVES / asset
    if archive.exists() and asset in ARCHIVE_SIZE and archive.stat().st_size != ARCHIVE_SIZE[asset]:
        archive.unlink()  # Incomplete previous download
    if not archive.is_file():
        temp = archive.with_suffix(".downloading")
        temp.unlink(missing_ok=True)
        print("Downloading:", asset, flush=True)
        subprocess.run(["wget", "--tries=3", "-q", "--show-progress",
                        "-O", str(temp), RELEASE + asset], check=True)
        if asset in ARCHIVE_SIZE and temp.stat().st_size != ARCHIVE_SIZE[asset]:
            raise RuntimeError(f"Downloaded {asset} size is incorrect.")
        temp.rename(archive)
    password = getpass(f"Private archive password for {asset}: ")
    try:
        proc = subprocess.run(["7z", "x", "-y", "-p" + password,
                               "-o" + str(dest), str(archive)],
                              capture_output=True, text=True)
    finally:
        del password
    if proc.returncode:
        raise RuntimeError(f"Extraction failed for {asset}: incorrect password or damaged archive.\n"
                           + (proc.stdout + proc.stderr)[-1000:])
    if not expected.is_file():
        raise FileNotFoundError(f"Extracted {asset}, but missing {expected}")

CHECKPOINT = ROOT / "final10k" / "resume.last.pt"
DATA = ROOT / "dataset_transfer" / "hf_coding_knowledge_v2"

restore_encrypted("phi_final10k.7z", CHECKPOINT.parent, CHECKPOINT)
restore_encrypted("phi_dataset_rekey.7z", DATA.parent, DATA / "manifest.json")
manifest = json.loads((DATA / "manifest.json").read_text())
ck = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
assert ck.get("format") == "PhiStateMoE-v1", "Wrong checkpoint architecture"
assert ck.get("step") == 10000, f"Wrong step: {ck.get('step')}"
assert "optimizer_state" in ck, "This checkpoint lacks optimizer state"
assert ck.get("data_fingerprint") == manifest.get("data_fingerprint"), "Corpus mismatch"
assert ck["config"]["vocab"] == manifest["vocab"], "Tokenizer vocabulary mismatch"
assert (DATA / "tokenizer.model").is_file(), "Missing original tokenizer"
print("Checkpoint step:", ck["step"])
print("Validation BPT:", ck.get("best_validation_bpt"))
print("Optimizer preserved:", True)
print("Corpus fingerprint:", manifest["data_fingerprint"])
del ck
gc.collect()

# Also validate all required .bin files have expected lengths.
sys.path.insert(0, str(LAB))
from moe_train import load_data
_, arrays, _ = load_data(str(DATA))
print("Verified dataset splits:", sorted(arrays))
del arrays
# The dataset archive also includes an earlier snapshot. Keep the FINAL 10K checkpoint only.
(DATA.parent / "resume.last.pt").unlink(missing_ok=True)
print("\nREADY FOR L4 TRAINING", flush=True)

In [ ]:
# 2 — Resume REAL model + corpus, do a bounded 20-step CUDA pilot.
# The original 10K checkpoint stays unchanged.
import subprocess, sys, json, torch
from pathlib import Path

RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

def run_gpu_continuation(source: Path, output: Path, target: int, evaluate_every: int):
    latest = output.with_name(output.stem + ".last.pt")
    if latest.is_file():
        saved = torch.load(latest, map_location="cpu", weights_only=True)
        if saved.get("step") == target and saved.get("data_fingerprint") == manifest["data_fingerprint"]:
            print("Existing completed checkpoint:", latest, flush=True)
            return latest
        raise RuntimeError(f"Existing checkpoint is incomplete. Preserve it: {latest}")
    if output.exists():
        raise RuntimeError(f"Output already exists: {output}")
    command = [
        sys.executable, "-u", str(LAB / "pretrain_continue_10k.py"),
        "--checkpoint", str(source), "--data-dir", str(DATA),
        "--out", str(output), "--target-steps", str(target),
        "--eval-every", str(evaluate_every), "--eval-batches", "16",
        "--threads", "2", "--device", "cuda"
    ]
    logfile = output.with_suffix(".log")
    print("Running:", " ".join(command), flush=True)
    with logfile.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"GPU training exited {rc}; actual error is printed above and saved in {logfile}")
    assert latest.is_file(), "Training finished without a latest checkpoint"
    saved = torch.load(latest, map_location="cpu", weights_only=True)
    assert saved.get("step") == target
    assert saved.get("data_fingerprint") == manifest["data_fingerprint"]
    assert "optimizer_state" in saved
    print("PASS: resumable CUDA checkpoint saved at step", target, flush=True)
    return latest

PILOT_LATEST = run_gpu_continuation(
    CHECKPOINT, RUNS / "phi_10020.pt", target=10020, evaluate_every=20
)
print("\n20-STEP REAL-DATA L4 PILOT PASS", flush=True)

In [ ]:
# 3 — Continue the validated GPU checkpoint through step 10,250.
# Only run after the 20-step pilot passes.
FINAL_LATEST = run_gpu_continuation(
    PILOT_LATEST, RUNS / "phi_10250.pt", target=10250, evaluate_every=50
)
print("\nL4 CONTINUATION FINISHED:", FINAL_LATEST, flush=True)

In [ ]:
# 4 — Back up completed GPU weights to Google Drive BEFORE disconnecting Colab.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")
backup = Path("/content/drive/MyDrive/Quantonium/PhiState-L4")
backup.mkdir(parents=True, exist_ok=True)

for path in [FINAL_LATEST, RUNS / "phi_10250.pt",
             RUNS / "phi_10020.last.pt", RUNS / "phi_10250.log"]:
    if path.is_file():
        dest = backup / path.name
        shutil.copy2(path, dest)
        assert dest.stat().st_size == path.stat().st_size
        print("BACKED UP:", dest, dest.stat().st_size, "bytes", flush=True)
print("\nSAVED TO GOOGLE DRIVE:", backup, flush=True)